#### Topic classification and paragraph splitting with LLM 

In [1]:
import os,sys
sys.path.insert(0,'../libs')
from dotenv import load_dotenv
from utils import load_json
import openai
from llm_utils import BSAgent
import numpy as np
import pandas as pd
import re,json,copy
from tqdm import tqdm

/data/home/xiong/miniconda3/envs/gpt/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### Load API Key and run one example 

In [4]:
## load API Key
# Load environment variables from .env file
load_dotenv()
openai.api_key  = os.getenv("OPENAI_API_KEY")

# %%
llm_agent  = BSAgent(model="gpt-4o", 
                    temperature=0)
## just run one test, make sure the api works 
pt = {'System':'You are a helpful assistant.',
      'Human':'What is your name?'}
res = llm_agent.get_response_content(prompt_template=pt)
print(res)  

I don't have a personal name, but you can call me Assistant. How can I help you today?


#### Try one paragraph and see how well it works 

In [6]:
df = pd.read_csv('/root/data/Fund/CSR/llm_topic_results.csv')
df.rename(columns={"paragraph": "examples"}, inplace=True)
print(df.iloc[0]['llm_topic'])
print(df.iloc[0]['examples'])
test_input = df.iloc[0]

Fiscal Management
Additional, targeted fiscal support is warranted in 2022-2023. The 2022 Budget allocated RM23 billion (1.4 percent of GDP) out of the COVID Fund for COVID-related spending, with an increasing share spent on upskilling programs, social assistance support to vulnerable groups, and SME financing. Considering the still-sizable projected economic slack, medium-term pandemic scarring effects, generally favorable debt dynamics, and staff’s assessment that debt is sustainable (Appendix II), staff advice is to provide modest additional fiscal support by about 2 percentage points cumulatively in 2022 and 2023 above the 2022 budget plans, moving the fiscal impulse in 2022 from negative to moderately positive territory, and delaying the start of consolidation by one year to 2023.8 The additional stimulus would close the output gap faster than in the baseline, thus helping limit the pandemic’s scarring effects. Targeted additional spending should center on protecting the vulnerabl

#### Run one basic economic sector question and see if LLM knows basic macro

In [11]:
simple_pt = {'System':'You are an experience macroeconomist from IMF. Your job is to assign topic labels to a given paragraph from IMF document',
      'Human':'Do you kne the difference between Real sector, Fiscal sector, Monetary sector, Financial sector and External sector ?'}

res = llm_agent.get_response_content(prompt_template=simple_pt)
print(res)  

Yes, I can explain the differences between these sectors:

1. **Real Sector**:
   - **Definition**: The real sector encompasses the production and consumption of goods and services in an economy. It includes activities related to agriculture, manufacturing, services, and trade.
   - **Key Indicators**: GDP, industrial production, employment rates, and productivity.

2. **Fiscal Sector**:
   - **Definition**: The fiscal sector involves government revenue and expenditure. It includes taxation, government spending, budget deficits/surpluses, and public debt.
   - **Key Indicators**: Government budget balance, public debt-to-GDP ratio, tax revenue, and government expenditure.

3. **Monetary Sector**:
   - **Definition**: The monetary sector deals with the supply of money, interest rates, and the overall monetary policy managed by a country's central bank.
   - **Key Indicators**: Money supply (M1, M2, etc.), interest rates, inflation rates, and central bank policy rates.

4. **Financial Se

In [21]:
topic_identify_simple_pt = {
    'System':'You are an experience macroeconomist from IMF. Your job is to assign topic labels to a given paragraph from IMF document',
    'Human':"""

You are given a list of topics with their definition and key indicators as below:
----------------
----------------
1. **Real Sector**:
   - **Definition**: The real sector encompasses the production and consumption of goods and services in an economy. It includes activities related to agriculture, manufacturing, services, and trade.
   - **Key Indicators**: GDP, industrial production, employment rates, and productivity.

2. **Fiscal Sector**:
   - **Definition**: The fiscal sector involves government revenue and expenditure. It includes taxation, government spending, budget deficits/surpluses, and public debt.
   - **Key Indicators**: Government budget balance, public debt-to-GDP ratio, tax revenue, and government expenditure.

3. **Monetary Sector**:
   - **Definition**: The monetary sector deals with the supply of money, interest rates, and the overall monetary policy managed by a country's central bank.
   - **Key Indicators**: Money supply (M1, M2, etc.), interest rates, inflation rates, and central bank policy rates.

4. **Financial Sector**:
   - **Definition**: The financial sector includes institutions and markets that facilitate the flow of funds between savers and borrowers. It encompasses banks, stock markets, bond markets, and other financial intermediaries.
   - **Key Indicators**: Stock market indices, bond yields, bank lending rates, and financial stability indicators.

5. **External Sector**:
   - **Definition**: The external sector covers a country's international economic transactions, including trade in goods and services, cross-border investment, and foreign exchange markets.
   - **Key Indicators**: Trade balance, current account balance, foreign direct investment (FDI), exchange rates, and international reserves.
----------------
----------------
    
You are also given a paragraph from a report published by the International Monetary Fund as below: 
----------------
----------------
{PARA}
----------------
----------------

Please carefully analyze the paragraph and classify the provided paragraph using ONLY the provided topics. 
Try your best to assign only one topic to the paragraph. You can use multiple categories only if you are very confident that multiple topics are extensively discussed in the paragraph.
If the paragraph does not fit into any of the provided you can return "None". 
Please provide your reasoning for your classification first, and then provide the topic label and a confidence score from 0-100.

Please respond in clean json format as follow:
reasoning: <explanation for topic label >,
topic_labels: [<topic label: confidence score>,...] or "None",
Make sure your answer starts with ```json

"""}

In [22]:
#print(topic_identify_simple_pt['Human'].format(PARA=test_input['examples']))
topic_pt_temp = copy.copy(topic_identify_simple_pt)
topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=test_input['examples'])
res = llm_agent.get_response_content(prompt_template=topic_pt_temp)
res_dict = llm_agent.parse_load_json_str(res) 
print(res_dict)

{'reasoning': 'The paragraph primarily discusses government revenue and expenditure, focusing on fiscal support measures, budget allocations, public debt, and fiscal consolidation. It mentions specific fiscal indicators such as the budget allocation for COVID-related spending, the public debt-to-GDP ratio, and the need for additional fiscal support. The emphasis is on fiscal policy actions and their implications for economic recovery and debt sustainability.', 'topic_labels': [{'Fiscal Sector': 95}]}


In [23]:
sample_results = []
for idx,r in tqdm(df.head(40).iterrows()):
    topic_pt_temp = copy.copy(topic_identify_simple_pt)
    topic_pt_temp['Human']=topic_identify_simple_pt['Human'].format(PARA=r['examples'])
    res = llm_agent.get_response_content(prompt_template=topic_pt_temp)
    res_dict = llm_agent.parse_load_json_str(res) 
    sample_results.append([r['llm_topic'],r['examples'],res_dict['reasoning'],res_dict['topic_labels']])

40it [01:10,  1.76s/it]


In [24]:
res_df = pd.DataFrame(sample_results,columns=['llm_topic','paragraph','reasoning','topic_labels'])
res_df.head()

,llm_topic,paragraph,reasoning,topic_labels
0,Fiscal Management,"Additional, targeted fiscal support is warrant...",The paragraph primarily discusses government r...,[{'Fiscal Sector': 95}]
1,Fiscal Management,The authorities agreed that additional fiscal ...,The paragraph primarily discusses government f...,[{'Fiscal Sector': 95}]
2,Fiscal Management,Box 1. Jordan: Past Fund Staff Advice and Impl...,The paragraph discusses various aspects of Jor...,[{'Fiscal Sector': 90}]
3,Financial Stability,The authorities consider that geopolitical ten...,The paragraph primarily discusses the stabilit...,[{'Financial Sector': 95}]
4,Financial Stability,The banking sector is in good financial health...,The paragraph primarily discusses the health a...,[{'Financial Sector': 95}]


In [27]:
res_df.to_csv('/root/data/Fund/CSR/llm_topic_results_v2.csv')